# P1 model performance — multi-seed figure

Redraws `output/all-model-performance.pdf` from the runs produced by
`run_multiseed_p1.py`, with mean ± std bands over seeds instead of the single
run in `KAN_2_LTN_hierarchy.ipynb`.

Reads only from `./p1_multiseed/` — **no training happens here**, so styling can
be retuned freely.

Preserved from the original figure: the colour and marker of every one of the
four original curves. Changed: the model names, which now follow the revised
macros in `main.tex` / `Major-Revision-Response.tex`.

| original label | revised label | macro | colour | marker |
|---|---|---|---|---|
| MLP | MLP | — | `#005f73` | `o` |
| LogiKNet | Logic-KAN | `\logickan` | `#fad02c` | `d` |
| H-LogiKNet | H-Logic-KAN | `\hlogickan` | `#7678ed` | `p` |
| H-LogiKNet+ | H-Logic-KAN* | `\hlogickanp` | `#780000` | `s` |
| — | Logic-MLP *(new)* | — | `#0a9396` | `^` |
| — | KAN (no logic) *(new)* | — | `#ee9b00` | `v` |

The two new colours were picked from the same palette family as the curves they
sit next to (`#0a9396` beside MLP's `#005f73`; `#ee9b00` beside Logic-KAN's
`#fad02c`), so an MLP-family and a KAN-family curve stay visually grouped.

In [ ]:
# --- Setup ---
import os, json, glob
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

RESULT_DIR = './p1_multiseed'
OUT_DIR    = './output'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.isdir(RESULT_DIR):
    raise FileNotFoundError(
        f'{RESULT_DIR} not found. Run the harness first:\n'
        '    python run_multiseed_p1.py            # full sweep\n'
        '    python run_multiseed_p1.py --fast     # 5-epoch wiring check')
print('runs found:', len(glob.glob(os.path.join(RESULT_DIR, 'curve__*.npz'))))

In [ ]:
# --- Model registry: names, colours, markers ---
# Order here is the plotting/legend order.
PLT = {
    'mlp':              dict(name='MLP',            color='#005f73', marker='o'),
    'logic_mlp':        dict(name='Logic-MLP',      color='#0a9396', marker='^'),
    'kan':              dict(name='KAN (no logic)', color='#ee9b00', marker='v'),
    'logic_kan':        dict(name='Logic-KAN',      color='#fad02c', marker='d'),
    'h_logic_kan':      dict(name='H-Logic-KAN',    color='#7678ed', marker='p'),
    'h_logic_kan_star': dict(name='H-Logic-KAN*',   color='#780000', marker='s'),
}

# Which curves go in the figure. Set to the original four to reproduce
# Fig. P1-all-model exactly; use all six for the response-letter ablation.
SHOW = ['mlp', 'logic_mlp', 'kan', 'logic_kan', 'h_logic_kan', 'h_logic_kan_star']
# SHOW = ['mlp', 'logic_kan', 'h_logic_kan', 'h_logic_kan_star']   # original four

In [ ]:
# --- Load curves: [n_seeds, n_epochs] per model per quantity ---
def load_curves(key):
    files = sorted(glob.glob(os.path.join(RESULT_DIR, f'curve__{key}__seed*.npz')))
    if not files:
        return None
    seeds, stacks = [], {}
    for f in files:
        seeds.append(int(f.split('seed')[-1].split('.')[0]))
        d = np.load(f)
        for k in d.files:
            stacks.setdefault(k, []).append(d[k])
    n = min(len(a) for a in stacks['epoch'])      # guard against short runs
    out = {k: np.stack([a[:n] for a in v]) for k, v in stacks.items()}
    out['seeds'] = np.array(seeds)
    return out


CURVES = {k: load_curves(k) for k in PLT}
CURVES = {k: v for k, v in CURVES.items() if v is not None}

for k, v in CURVES.items():
    acc = v['test_acc'][:, -1]
    print(f"{PLT[k]['name']:<16} seeds {list(v['seeds'])} | "
          f"{v['epoch'].shape[1]} epochs | "
          f"final acc {acc.mean():.4f} +/- {acc.std(ddof=1) if len(acc)>1 else 0:.4f}")

missing = [k for k in SHOW if k not in CURVES]
if missing:
    print('\nWARNING: no runs on disk for', missing, '-- they will be skipped.')
SHOW = [k for k in SHOW if k in CURVES]

---
## Figure config

Tune here, then re-run the figure cell.

In [ ]:
# --- Figure config ---
from matplotlib import font_manager

FONT_STACK = ['Times New Roman', 'Times', 'TeX Gyre Termes',
              'Nimbus Roman', 'Liberation Serif', 'DejaVu Serif']

def resolve_font(stack):
    avail = {f.name for f in font_manager.fontManager.ttflist}
    for n in stack:
        if n in avail:
            return n
    return None

FONT = resolve_font(FONT_STACK)
if FONT is None:
    FONT = 'DejaVu Serif'
    print('WARNING: none of', FONT_STACK, 'installed; using DejaVu Serif.')
elif FONT != FONT_STACK[0]:
    print(f'NOTE: "{FONT_STACK[0]}" not installed -> using "{FONT}".')
else:
    print('font:', FONT)

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': [FONT],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    # 3 = Type 3 (matplotlib default, REJECTED by IEEE PDF eXpress)
    # 42 = TrueType. Always 42.
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'axes.linewidth': 0.8,
})

FIGSIZE     = (15, 4)     # matches the original 1x3 layout
FONT_SIZE   = 14          # axis labels + ticks (original)
FONT_LEGEND = 12          # legend (original font_size_axis)
LINE_WIDTH  = 2           # original
MARKEVERY   = 20          # original
BAND_ALPHA  = 0.18        # shaded +/- 1 std
SHOW_BAND   = True
SAVE_FORMATS = ['pdf', 'png']
FIG_NAME    = 'all-model-performance-multiseed'


def band(ax, x, Y, color, label=None, marker=None, linestyle='-'):
    """Mean line over seeds with a +/-1 std ribbon.

    Y is [n_seeds, n_epochs]; all-NaN columns (e.g. satisfaction for a
    cross-entropy model) are dropped so nothing is plotted for them.
    """
    if Y is None or np.all(np.isnan(Y)):
        return
    mu = np.nanmean(Y, axis=0)
    sd = np.nanstd(Y, axis=0, ddof=1) if Y.shape[0] > 1 else np.zeros_like(mu)
    ax.plot(x, mu, color=color, label=label, linewidth=LINE_WIDTH,
            linestyle=linestyle, marker=marker, markevery=MARKEVERY,
            markersize=5)
    if SHOW_BAND and Y.shape[0] > 1:
        ax.fill_between(x, mu - sd, mu + sd, color=color, alpha=BAND_ALPHA,
                        linewidth=0)


def save_fig(fig, name):
    import shutil, subprocess
    for ext in SAVE_FORMATS:
        p = os.path.join(OUT_DIR, f'{name}.{ext}')
        fig.savefig(p, format=ext)
        if ext == 'pdf':
            if shutil.which('pdffonts'):
                out = subprocess.run(['pdffonts', p], capture_output=True,
                                     text=True).stdout
                bad = [l for l in out.splitlines() if 'Type 3' in l]
                if bad:
                    raise RuntimeError('Type 3 font in ' + p + ':\n' + '\n'.join(bad))
            elif b'/Type3' in open(p, 'rb').read():
                raise RuntimeError('Type 3 font detected in ' + p)
    print('saved:', ', '.join(os.path.join(OUT_DIR, f'{name}.{e}')
                              for e in SAVE_FORMATS), '| no Type 3 fonts')

### The figure

Same three panels as the original: training loss, satisfaction, test accuracy.
Solid line = mean over seeds, shaded ribbon = ±1 std.

Note the loss panel mixes two different objectives — cross-entropy for MLP and
KAN (no logic), `1 − sat` for the four LTN-trained models. That was true of the
original figure too; it's worth a sentence in the caption so a reviewer doesn't
read the gap as a fair loss comparison.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=FIGSIZE)

# --- (a) training loss -----------------------------------------------------
for k in SHOW:
    c, p = CURVES[k], PLT[k]
    band(axs[0], c['epoch'][0], c['train_loss'], p['color'], label=p['name'])
axs[0].set_xlabel('Epoch', fontsize=FONT_SIZE)
axs[0].set_ylabel('Loss', fontsize=FONT_SIZE)
axs[0].tick_params(axis='both', labelsize=FONT_SIZE)
axs[0].legend(prop={'size': FONT_LEGEND})

# --- (b) satisfaction (LTN-trained models only) -----------------------------
for k in SHOW:
    c, p = CURVES[k], PLT[k]
    if np.all(np.isnan(c['train_sat'])):
        continue                      # cross-entropy model: no satisfaction
    band(axs[1], c['epoch'][0], c['train_sat'], p['color'], label=p['name'],
         linestyle='--')
axs[1].set_xlabel('Epoch', fontsize=FONT_SIZE)
axs[1].set_ylabel('Satisfaction', fontsize=FONT_SIZE)
axs[1].tick_params(axis='both', labelsize=FONT_SIZE)
axs[1].legend(loc='lower right', prop={'size': FONT_LEGEND})

# --- (c) test accuracy ------------------------------------------------------
for k in SHOW:
    c, p = CURVES[k], PLT[k]
    band(axs[2], c['epoch'][0], c['test_acc'], p['color'], label=p['name'],
         marker=p['marker'])
axs[2].set_xlabel('Epoch', fontsize=FONT_SIZE)
axs[2].set_ylabel('Accuracy', fontsize=FONT_SIZE)
axs[2].tick_params(axis='both', labelsize=FONT_SIZE)
axs[2].legend(prop={'size': FONT_LEGEND})

plt.tight_layout()
save_fig(fig, FIG_NAME)
plt.show()

### LaTeX tables for the response letter

Emits the ablation table (predictive + calibration) and the hierarchical table
straight from `summary.json` / `hierarchical.json`, using the revised macros so
nothing has to be retyped.

In [ ]:
MACRO = {
    'mlp': 'MLP', 'logic_mlp': 'Logic-MLP', 'kan': 'KAN (no logic)',
    'logic_kan': r'\logickan{}', 'h_logic_kan': r'\hlogickan{}',
    'h_logic_kan_star': r'\hlogickanp{}',
}
TICK = {'mlp': ('--', '--', '--'), 'logic_mlp': ('--', '--', r'\checkmark'),
        'kan': ('--', r'\checkmark', '--'),
        'logic_kan': ('--', r'\checkmark', r'\checkmark'),
        'h_logic_kan': (r'\checkmark', r'\checkmark', r'\checkmark'),
        'h_logic_kan_star': (r'\checkmark', r'\checkmark', r'\checkmark')}

with open(os.path.join(RESULT_DIR, 'summary.json')) as f:
    S = json.load(f)

def cell(row, m, fmt='{:.3f}'):
    if m not in row:
        return '--'
    return f"${fmt.format(row[m]['mean'])}\\pm{fmt.format(row[m]['std'])}$"

cols = [('accuracy', 'Accuracy'), ('macro_f1', 'Macro-F1'),
        ('macro_recall', 'Macro Rec.'), ('macro_fpr', 'Macro FPR'),
        ('macro_auroc', 'AUROC'), ('ece', 'ECE')]

print(r'\begin{tabular}{l' + 'c' * (3 + len(cols)) + '}')
print(r'\toprule')
print('Model & Hier. & KAN & Logic & ' +
      ' & '.join(h for _, h in cols) + r' \\')
print(r'\midrule')
for k in [m for m in PLT if m in S]:
    h, kn, lg = TICK[k]
    print(f'{MACRO[k]} & {h} & {kn} & {lg} & ' +
          ' & '.join(cell(S[k], m) for m, _ in cols) + r' \\')
print(r'\bottomrule')
print(r'\end{tabular}')

# --- hierarchical table -----------------------------------------------------
with open(os.path.join(RESULT_DIR, 'hierarchical.json')) as f:
    H = json.load(f)
partials = H['partials']

print('\n\n% --- hierarchical evaluation (separate table) ---')
print(r'\begin{tabular}{l' + 'c' * (len(partials) + 1) + '}')
print(r'\toprule')
print('Model & ' + ' & '.join(f'$R_{{w={p}}}$' for p in partials) +
      r' & Hier.\ F1 \\')
print(r'\midrule')
for k in [m for m in PLT if m in H['models']]:
    r = H['models'][k]
    vals = [cell(r, f'reliability@{p}') for p in partials]
    print(f'{MACRO[k]} & ' + ' & '.join(vals) + ' & ' +
          cell(r, 'hierarchical_f1') + r' \\')
print(r'\bottomrule')
print(r'\end{tabular}')
print('\n% NOTE: R_{w=0.5} == Hier. F1 by construction for a single-parent')
print('%       two-level tree -- state this rather than presenting them as')
print('%       independent corroboration.')